In [1]:
import numpy as np
import matplotlib.pyplot as plt

Assume that the numerical solution has been computed on coarse and fine grid levels,  where the grid refinement is by a factor of two in each coordinate direction, using a  formally third-order accurate discretization. If the numerical solutions on the coarse and  fine grid are 20 and 23.5, respectively, use Richardson extrapolation to estimate the  exact solution to the differential equation. What major assumption must you make? 

Oberkampf, William L.; Roy, Christopher J.. Verification, Validation, and Uncertainty Quantification in Scientific Computing (p. 321). (Function). Kindle Edition. 

In [42]:
def compute_richardson_extrapolation(grid, r, p):
    if len(grid)==2:
        fine=grid[0]
        coarse=grid[1]
        fexact=fine+((fine-coarse)/((r**p)-1))
    return fexact


In [44]:
compute_richardson_extrapolation([1.2, 1.25], 2, 2)

1.1833333333333333

In [16]:
compute_richardson_extrapolation([23.5, 20], 2, 3)

24.0


24.0

In [26]:
def compute_observed_order_of_accuracy(grid, r):
    if len(grid)==2:
        fine=grid[0]
        coarse=grid[1]
        phat=(np.log(coarse/fine))/np.log(r)
        return phat
    elif len(grid)==3:
        fine=grid[0]
        medium=grid[1]
        coarse=grid[2]
        phat=(np.log((coarse-medium)/(medium-fine)))/np.log(r)
        return phat

In [40]:
compute_observed_order_of_accuracy([1.2, 1.25, 1.45], 2)

np.float64(1.9999999999999984)

In [32]:
def compute_gci(grid, r, phat, pf):
    factor_of_safety = np.abs((phat-pf)/(pf))
    if factor_of_safety <= 0.1:
        print("The observed order of accuracy is within 10% of the predicted order of accuracy.")
        fs=1.25
        print("The factor of safety is: ", fs)
    else:
        print("The observed order of accuracy is not within 10% of the predicted order of accuracy.")
        fs=3.0
        print("The factor of safety is: ", fs)
    fine=grid[0]
    coarse=grid[1]
    gci=((fs)/((r**phat)-1))*np.abs(coarse-fine)
    return gci



In [30]:
def perform_solution_verification(grid, r, pf):
    phat=compute_observed_order_of_accuracy(grid, r)
    print("The observed order of accuracy is: ", phat)
    richardson_exp=compute_richardson_extrapolation(grid[0:2], r, phat)
    print("The Richardson extrapolation is: ", richardson_exp)
    gci=compute_gci(grid[0:2], r, phat, pf)
    print("The Grid Convergence Index is: ", gci)
    print(f"The interval is {grid[0]} +/- {gci}")
    

In [45]:
perform_solution_verification([1.2, 1.25,1.45], 2, 2)

The observed order of accuracy is:  1.9999999999999984
The Richardson extrapolation is:  1.1833333333333333
The observed order of accuracy is within 10% of the predicted order of accuracy.
The factor of safety is:  1.25
The Grid Convergence Index is:  0.020833333333333384
The interval is 1.2 +/- 0.020833333333333384


In [47]:
1.2+0.021

1.2209999999999999

In [33]:
perform_solution_verification([1.2, 1.26,1.43], 2, 2)

The observed order of accuracy is:  1.5025003405291812
The Richardson extrapolation is:  1.167272727272727
The observed order of accuracy is not within 10% of the predicted order of accuracy.
The factor of safety is:  3.0
The Grid Convergence Index is:  0.09818181818181848
The interval is 1.2 +/- 0.09818181818181848


In [36]:
grid=[5.1, 5.4,6.35]
r=2
p=2

compute_gci(grid[0:2], r, phat=compute_observed_order_of_accuracy(grid, r), pf=p)

The observed order of accuracy is not within 10% of the predicted order of accuracy.
The factor of safety is:  3.0


np.float64(0.4153846153846182)

In [51]:
def area_validation_metric(exp, sim):

    exp = sorted(exp)
    sim = sorted(sim)

    values = sorted(set(exp + sim))
    print(values)

    total_area = 0.0
    print([(i, j) for i, j in zip(values[:-1], values[1:])])
    for left, right in zip(values[:-1], values[1:]):
        # Empirical CDF values on the interval [left, right)
        f_exp = sum(x <= left for x in exp) / len(exp)
        print(f"f_exp for interval [{left}, {right}): {f_exp}")
        f_sim = sum(x <= left for x in sim) / len(sim)
        print(f"f_sim for interval [{left}, {right}): {f_sim}")

        width = right - left
        print(f"Width of interval [{left}, {right}): {width}")
        total_area += abs(f_exp - f_sim) * width
        print(f"Area contribution from interval [{left}, {right}): {abs(f_exp - f_sim) * width}")
        print("\n")

    return total_area
exp = [8.1, 8.3, 8.7, 8.3]
sim = [8.2]

area = area_validation_metric(exp, sim)
print(area)

[8.1, 8.2, 8.3, 8.7]
[(8.1, 8.2), (8.2, 8.3), (8.3, 8.7)]
f_exp for interval [8.1, 8.2): 0.25
f_sim for interval [8.1, 8.2): 0.0
Width of interval [8.1, 8.2): 0.09999999999999964
Area contribution from interval [8.1, 8.2): 0.02499999999999991


f_exp for interval [8.2, 8.3): 0.25
f_sim for interval [8.2, 8.3): 1.0
Width of interval [8.2, 8.3): 0.10000000000000142
Area contribution from interval [8.2, 8.3): 0.07500000000000107


f_exp for interval [8.3, 8.7): 0.75
f_sim for interval [8.3, 8.7): 1.0
Width of interval [8.3, 8.7): 0.3999999999999986
Area contribution from interval [8.3, 8.7): 0.09999999999999964


0.20000000000000062


In [59]:
def modified_area_validation_metric(exp, sim):
    """
    Compute modified area validation metric between two empirical CDFs.

    Returns
    -------
    dict with:
        d_plus  : area where F_sim > F_exp
        d_minus : area where F_exp > F_sim
        d_total : d_plus + d_minus
    """

    if len(exp) == 0 or len(sim) == 0:
        raise ValueError("Both input lists must contain at least one value.")

    exp = sorted(exp)
    sim = sorted(sim)

    values = sorted(set(exp + sim))

    d_plus = 0.0
    d_minus = 0.0

    for left, right in zip(values[:-1], values[1:]):
        f_exp = sum(x <= left for x in exp) / len(exp)
        f_sim = sum(x <= left for x in sim) / len(sim)

        width = right - left
        diff = f_sim - f_exp
        print(f"Interval [{left}, {right}): f_exp={f_exp}, f_sim={f_sim}, diff={diff}, width={width}")
        if diff > 0:
            d_plus += diff * width
        elif diff < 0:
            d_minus += abs(diff) * width

    return {
        "d_plus": d_plus,
        "d_minus": d_minus,
        "d_total": d_plus + d_minus
    }
exp = [8.1, 8.3, 8.7, 8.3]
sim = [8.2]

result = modified_area_validation_metric(exp, sim)
print(result)

Interval [8.1, 8.2): f_exp=0.25, f_sim=0.0, diff=-0.25, width=0.09999999999999964
Interval [8.2, 8.3): f_exp=0.25, f_sim=1.0, diff=0.75, width=0.10000000000000142
Interval [8.3, 8.7): f_exp=0.75, f_sim=1.0, diff=0.25, width=0.3999999999999986
{'d_plus': 0.1750000000000007, 'd_minus': 0.02499999999999991, 'd_total': 0.20000000000000062}


In [58]:
.2-0.02499999999999991


0.1750000000000001

In [52]:
def modified_area_validation_metric(exp, sim):
    """
    Directional empirical-CDF area metric.

    d_minus = area where F_exp > F_sim
    d_plus  = area where F_sim > F_exp
    """

    if not exp or not sim:
        raise ValueError("Both samples must be non-empty.")

    exp = sorted(exp)
    sim = sorted(sim)

    values = sorted(set(exp + sim))

    d_minus = 0.0
    d_plus = 0.0

    for left, right in zip(values[:-1], values[1:]):
        f_exp = sum(x <= left for x in exp) / len(exp)
        f_sim = sum(x <= left for x in sim) / len(sim)

        width = right - left
        diff = f_exp - f_sim

        if diff > 0:
            d_minus += diff * width
        elif diff < 0:
            d_plus += abs(diff) * width

    return d_minus, d_plus
exp = [8.1, 8.3, 8.7, 8.3]
sim = [8.2]

modified_area_validation_metric(exp, sim)


(0.02499999999999991, 0.1750000000000007)

In [ ]:
from tqdm import tqdm
from tqdm.notebook import tqdm
tqdm.pandas()

ne10_samples = np.linspace(0.08, 0.12, 10 + 1)
ne10_samples = np.linspace(0.08, 0.12, 10 + 1)
total_srq=[]
seedval=41
for k in range(0, len(ne10_samples)):
    SRQ50=[]
    print(f"Running for upc={ne10_samples[k]:.4f}")
    sampler50 = qmc.LatinHypercube(d=1, seed=seedval+k)
    sample50= sampler50.random(n=50)
    sample50 = sample50.flatten()
    sample50 = norm.ppf(sample50, loc=1.25, scale=0.25)
    sample50 = np.clip(sample50, 1e-6, None)
    print("Starting simulations...")
    for i in tqdm(range(len(sample50)), desc="Simulated samples", leave=False):
        physical_parameters = {    
            'kap': ne10_samples[k], # diffusion coefficient
            'eps': eps, # inverse of activation energy
            'upc': sample50[i], # u phase change
            'q': q, # reaction heat
            'alp': alp, 
            'x_lim': (x_min, x_max), # x-axis domain 
            'y_lim': (y_min, y_max), # y-axis domain
            't_lim': (t_min, t_max), # time domain
            'components':(True, False, True) #diffusion, convection, reaction
        }
        SRQ50.append(runSolver(physical_parameters)) 
    total_srq.append(SRQ50)
total_srq = np.array(total_srq)
print(total_srq.shape)